This project focuses on tree-based classification models.

The goal is to predict whether a car is a bad buy using the `IsBadBuy` target.

The project starts with a custom Decision Tree implementation, then extends the idea to ensemble models such as Random Forest and GBDT. Modern boosting libraries such as LightGBM, CatBoost, and XGBoost will also be tested and compared.

The dataset is split by `PurchDate` to avoid data leakage. The oldest 33% of the data is used for training, the middle 33% for validation, and the newest 33% for testing.

using Data from: https://www.kaggle.com/competitions/DontGetKicked/data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/training.csv")

df["PurchDate"] = pd.to_datetime(df["PurchDate"])
df = df.sort_values("PurchDate").reset_index(drop=True)

n = len(df)

train_end = int(n * 0.33)
valid_end = int(n * 0.66)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

target = "IsBadBuy"
drop_columns = ["IsBadBuy", "RefId", "PurchDate"]

X_train = train_df.drop(columns=drop_columns)
y_train = train_df[target]

X_valid = valid_df.drop(columns=drop_columns)
y_valid = valid_df[target]

X_test = test_df.drop(columns=drop_columns)
y_test = test_df[target]

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)

print("\nNumeric:", len(numeric_features))
print(numeric_features)

print("\nCategorical:", len(categorical_features))
print(categorical_features)

X_train: (24084, 31)
X_valid: (24084, 31)
X_test: (24815, 31)

Numeric: 17
['VehYear', 'VehicleAge', 'WheelTypeID', 'VehOdo', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'BYRNO', 'VNZIP1', 'VehBCost', 'IsOnlineSale', 'WarrantyCost']

Categorical: 14
['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST']


Numeric and categorical features were separated.  
Categorical columns will be encoded using only the training data to avoid data leakage.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_valid_prepared = preprocessor.transform(X_valid)
X_test_prepared = preprocessor.transform(X_test)

print("Train prepared:", X_train_prepared.shape)
print("Valid prepared:", X_valid_prepared.shape)
print("Test prepared:", X_test_prepared.shape)

Train prepared: (24084, 1705)
Valid prepared: (24084, 1705)
Test prepared: (24815, 1705)


## Sklearn Decision Tree Baseline

A sklearn Decision Tree model was trained first as a baseline before implementing a custom tree from scratch.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

sk_tree = DecisionTreeClassifier(
    max_depth=7,
    random_state=21
)

sk_tree.fit(X_train_prepared, y_train)

valid_proba_tree = sk_tree.predict_proba(X_valid_prepared)[:, 1]

valid_auc_tree = roc_auc_score(y_valid, valid_proba_tree)
valid_gini_tree = 2 * valid_auc_tree - 1

print("Validation ROC AUC:", valid_auc_tree)
print("Validation Gini:", valid_gini_tree)

Validation ROC AUC: 0.6475046693384456
Validation Gini: 0.2950093386768913


The sklearn Decision Tree baseline achieved a validation Gini score of `0.2950`.

This is a strong baseline and will be used later to compare against the custom Decision Tree implementation.

In [4]:
X_train_tree = X_train[numeric_features].copy()
X_valid_tree = X_valid[numeric_features].copy()

num_imputer_tree = SimpleImputer(strategy="median")

X_train_tree = num_imputer_tree.fit_transform(X_train_tree)
X_valid_tree = num_imputer_tree.transform(X_valid_tree)

y_train_tree = y_train.values
y_valid_tree = y_valid.values

print("Train numeric:", X_train_tree.shape)
print("Valid numeric:", X_valid_tree.shape)

Train numeric: (24084, 17)
Valid numeric: (24084, 17)


## Custom Decision Tree

A custom Decision Tree classifier will be implemented using numeric features only.

The tree will use Gini impurity to choose the best split.

In [5]:
class Node:
    def __init__(self, depth=0):
        self.depth = depth
        self.left = None
        self.right = None
        self.feature_index = None
        self.threshold = None
        self.proba = None
        self.is_leaf = False

    def gini(self, y):
        if len(y) == 0:
            return 0
        
        p1 = np.mean(y)
        p0 = 1 - p1
        
        return 1 - p0**2 - p1**2

In [6]:
class CustomDecisionTreeClassifier:
    def __init__(self, max_depth=7, min_samples_split=50, max_thresholds=20):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_thresholds = max_thresholds

    def fit(self, X, y):
        self.root = self._build_tree(X, y, depth=0)
        return self

    def _best_split(self, X, y):
        best_feature = None
        best_threshold = None
        best_gini = float("inf")

        n_samples, n_features = X.shape

        for feature_index in range(n_features):
            values = X[:, feature_index]
            unique_values = np.unique(values)

            if len(unique_values) > self.max_thresholds:
                thresholds = np.percentile(
                    unique_values,
                    np.linspace(5, 95, self.max_thresholds)
                )
            else:
                thresholds = unique_values

            for threshold in thresholds:
                left_mask = values <= threshold
                right_mask = values > threshold

                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]

                left_gini = Node().gini(y_left)
                right_gini = Node().gini(y_right)

                weighted_gini = (
                    len(y_left) / n_samples * left_gini +
                    len(y_right) / n_samples * right_gini
                )

                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = feature_index
                    best_threshold = threshold

        return best_feature, best_threshold

    def _build_tree(self, X, y, depth):
        node = Node(depth=depth)
        node.proba = np.mean(y)

        if (
            depth >= self.max_depth or
            len(y) < self.min_samples_split or
            len(np.unique(y)) == 1
        ):
            node.is_leaf = True
            return node

        feature_index, threshold = self._best_split(X, y)

        if feature_index is None:
            node.is_leaf = True
            return node

        node.feature_index = feature_index
        node.threshold = threshold

        left_mask = X[:, feature_index] <= threshold
        right_mask = X[:, feature_index] > threshold

        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return node

    def _predict_one_proba(self, x, node):
        if node.is_leaf:
            return node.proba

        if x[node.feature_index] <= node.threshold:
            return self._predict_one_proba(x, node.left)
        else:
            return self._predict_one_proba(x, node.right)

    def predict_proba(self, X):
        probabilities = np.array([
            self._predict_one_proba(x, self.root)
            for x in X
        ])

        return np.column_stack([1 - probabilities, probabilities])

    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= threshold).astype(int)

In [7]:
# test

custom_tree = CustomDecisionTreeClassifier(
    max_depth=7,
    min_samples_split=50,
    max_thresholds=20
)

custom_tree.fit(X_train_tree, y_train_tree)

custom_tree_valid_proba = custom_tree.predict_proba(X_valid_tree)[:, 1]

custom_tree_auc = roc_auc_score(y_valid_tree, custom_tree_valid_proba)
custom_tree_gini = 2 * custom_tree_auc - 1

tree_comparison = pd.DataFrame({
    "model": ["Custom Decision Tree", "Sklearn Decision Tree"],
    "validation_roc_auc": [custom_tree_auc, valid_auc_tree],
    "validation_gini": [custom_tree_gini, valid_gini_tree]
})

tree_comparison.sort_values("validation_gini", ascending=False)

,model,validation_roc_auc,validation_gini
1,Sklearn Decision Tree,0.647505,0.295009
0,Custom Decision Tree,0.636033,0.272066


The custom Decision Tree achieved a validation Gini score of `0.2721`.

This is above the required minimum of `0.1`, so the custom implementation works successfully.

## Custom Random Forest

A custom Random Forest will be implemented using multiple custom Decision Trees.

Each tree will be trained on a random sample of rows and random subset of features.

In [8]:
class CustomRandomForestClassifier:
    def __init__(self, n_estimators=30, max_depth=7, min_samples_split=50,
                 max_thresholds=20, max_features="sqrt", random_state=21):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_thresholds = max_thresholds
        self.max_features = max_features
        self.random_state = random_state

    def fit(self, X, y):
        self.trees = []
        self.feature_indices = []
        
        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape
        
        if self.max_features == "sqrt":
            n_selected_features = int(np.sqrt(n_features))
        else:
            n_selected_features = self.max_features
        
        for _ in range(self.n_estimators):
            row_indices = rng.choice(n_samples, size=n_samples, replace=True)
            feature_indices = rng.choice(n_features, size=n_selected_features, replace=False)
            
            X_sample = X[row_indices][:, feature_indices]
            y_sample = y[row_indices]
            
            tree = CustomDecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_thresholds=self.max_thresholds
            )
            
            tree.fit(X_sample, y_sample)
            
            self.trees.append(tree)
            self.feature_indices.append(feature_indices)
        
        return self

    def predict_proba(self, X):
        tree_probs = []
        
        for tree, feature_indices in zip(self.trees, self.feature_indices):
            X_subset = X[:, feature_indices]
            proba = tree.predict_proba(X_subset)[:, 1]
            tree_probs.append(proba)
        
        avg_proba = np.mean(tree_probs, axis=0)
        return np.column_stack([1 - avg_proba, avg_proba])

    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= threshold).astype(int)

In [9]:
# test 

custom_rf = CustomRandomForestClassifier(
    n_estimators=30,
    max_depth=7,
    min_samples_split=50,
    max_thresholds=20,
    max_features="sqrt",
    random_state=21
)

custom_rf.fit(X_train_tree, y_train_tree)

custom_rf_valid_proba = custom_rf.predict_proba(X_valid_tree)[:, 1]

custom_rf_auc = roc_auc_score(y_valid_tree, custom_rf_valid_proba)
custom_rf_gini = 2 * custom_rf_auc - 1

print("Custom Random Forest ROC AUC:", custom_rf_auc)
print("Custom Random Forest Gini:", custom_rf_gini)

Custom Random Forest ROC AUC: 0.643754732212676
Custom Random Forest Gini: 0.28750946442535197


The custom Random Forest achieved a validation Gini score of `0.2875`.

This improves the custom single Decision Tree result (`0.2721`) and passes the required minimum Gini score of `0.15`.

The improvement comes from averaging multiple trees trained on random samples of rows and features.

In [10]:
# comprestion 

tree_ensemble_results = pd.DataFrame({
    "model": [
        "Custom Decision Tree",
        "Custom Random Forest",
        "Sklearn Decision Tree"
    ],
    "validation_roc_auc": [
        custom_tree_auc,
        custom_rf_auc,
        valid_auc_tree
    ],
    "validation_gini": [
        custom_tree_gini,
        custom_rf_gini,
        valid_gini_tree
    ]
})

tree_ensemble_results.sort_values("validation_gini", ascending=False)

,model,validation_roc_auc,validation_gini
2,Sklearn Decision Tree,0.647505,0.295009
1,Custom Random Forest,0.643755,0.287509
0,Custom Decision Tree,0.636033,0.272066


## Sklearn Random Forest

A sklearn Random Forest model was trained to compare its performance with the custom Random Forest implementation.

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

sk_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=7,
    min_samples_split=50,
    max_features="sqrt",
    random_state=21,
    n_jobs=-1
)

sk_rf.fit(X_train_prepared, y_train)

valid_proba_sk_rf = sk_rf.predict_proba(X_valid_prepared)[:, 1]

valid_auc_sk_rf = roc_auc_score(y_valid, valid_proba_sk_rf)
valid_gini_sk_rf = 2 * valid_auc_sk_rf - 1

print("Sklearn Random Forest ROC AUC:", valid_auc_sk_rf)
print("Sklearn Random Forest Gini:", valid_gini_sk_rf)

Sklearn Random Forest ROC AUC: 0.6586046039855634
Sklearn Random Forest Gini: 0.31720920797112684


In [12]:
# comparison table 

rf_comparison = pd.DataFrame({
    "model": [
        "Custom Decision Tree",
        "Custom Random Forest",
        "Sklearn Decision Tree",
        "Sklearn Random Forest"
    ],
    "validation_roc_auc": [
        custom_tree_auc,
        custom_rf_auc,
        valid_auc_tree,
        valid_auc_sk_rf
    ],
    "validation_gini": [
        custom_tree_gini,
        custom_rf_gini,
        valid_gini_tree,
        valid_gini_sk_rf
    ]
})

rf_comparison.sort_values("validation_gini", ascending=False)

,model,validation_roc_auc,validation_gini
3,Sklearn Random Forest,0.658605,0.317209
2,Sklearn Decision Tree,0.647505,0.295009
1,Custom Random Forest,0.643755,0.287509
0,Custom Decision Tree,0.636033,0.272066


Sklearn Random Forest achieved the best validation result among the tree models so far, with a Gini score of `0.3172`.

The custom Random Forest improved over the custom single Decision Tree, which shows that averaging multiple trees gives more stable predictions.

Sklearn implementations performed better than the custom ones because they use more optimized split search, training logic, and implementation details.

## Custom GBDT

A simple GBDT classifier will be implemented using custom regression trees.

Each new tree will learn from the residual errors of the current ensemble.

In [13]:
class CustomDecisionTreeRegressor:
    def __init__(self, max_depth=3, min_samples_split=50, max_thresholds=20):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_thresholds = max_thresholds

    def fit(self, X, y):
        self.root = self._build_tree(X, y, depth=0)
        return self

    def _best_split(self, X, y):
        best_feature = None
        best_threshold = None
        best_mse = float("inf")

        n_samples, n_features = X.shape

        for feature_index in range(n_features):
            values = X[:, feature_index]
            unique_values = np.unique(values)

            if len(unique_values) > self.max_thresholds:
                thresholds = np.percentile(
                    unique_values,
                    np.linspace(5, 95, self.max_thresholds)
                )
            else:
                thresholds = unique_values

            for threshold in thresholds:
                left_mask = values <= threshold
                right_mask = values > threshold

                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]

                mse = (
                    len(y_left) / n_samples * np.var(y_left) +
                    len(y_right) / n_samples * np.var(y_right)
                )

                if mse < best_mse:
                    best_mse = mse
                    best_feature = feature_index
                    best_threshold = threshold

        return best_feature, best_threshold

    def _build_tree(self, X, y, depth):
        node = Node(depth=depth)
        node.proba = np.mean(y)

        if (
            depth >= self.max_depth or
            len(y) < self.min_samples_split
        ):
            node.is_leaf = True
            return node

        feature_index, threshold = self._best_split(X, y)

        if feature_index is None:
            node.is_leaf = True
            return node

        node.feature_index = feature_index
        node.threshold = threshold

        left_mask = X[:, feature_index] <= threshold
        right_mask = X[:, feature_index] > threshold

        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return node

    def _predict_one(self, x, node):
        if node.is_leaf:
            return node.proba

        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

In [14]:
class CustomGBDTClassifier:
    def __init__(self, number_of_trees=30, learning_rate=0.1,
                 max_depth=3, min_samples_split=50,
                 max_thresholds=20, max_features=None,
                 random_state=21):
        self.number_of_trees = number_of_trees
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_thresholds = max_thresholds
        self.max_features = max_features
        self.random_state = random_state

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)

        self.trees = []
        self.feature_indices = []

        positive_rate = np.clip(np.mean(y), 1e-6, 1 - 1e-6)
        self.initial_prediction = np.log(positive_rate / (1 - positive_rate))

        raw_predictions = np.full(len(y), self.initial_prediction)

        rng = np.random.default_rng(self.random_state)
        n_features = X.shape[1]

        for _ in range(self.number_of_trees):
            probabilities = self._sigmoid(raw_predictions)

            residuals = y - probabilities

            if self.max_features is None:
                feature_indices = np.arange(n_features)
            elif self.max_features == "sqrt":
                feature_indices = rng.choice(
                    n_features,
                    size=int(np.sqrt(n_features)),
                    replace=False
                )
            else:
                feature_indices = rng.choice(
                    n_features,
                    size=self.max_features,
                    replace=False
                )

            tree = CustomDecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_thresholds=self.max_thresholds
            )

            tree.fit(X[:, feature_indices], residuals)

            raw_predictions += self.learning_rate * tree.predict(X[:, feature_indices])

            self.trees.append(tree)
            self.feature_indices.append(feature_indices)

        return self

    def predict_proba(self, X):
        X = np.array(X)

        raw_predictions = np.full(X.shape[0], self.initial_prediction)

        for tree, feature_indices in zip(self.trees, self.feature_indices):
            raw_predictions += self.learning_rate * tree.predict(X[:, feature_indices])

        probabilities = self._sigmoid(raw_predictions)

        return np.column_stack([1 - probabilities, probabilities])

    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= threshold).astype(int)

In [15]:
# test.. 

custom_gbdt = CustomGBDTClassifier(
    number_of_trees=30,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=50,
    max_thresholds=20,
    max_features="sqrt",
    random_state=21
)

custom_gbdt.fit(X_train_tree, y_train_tree)

custom_gbdt_valid_proba = custom_gbdt.predict_proba(X_valid_tree)[:, 1]

custom_gbdt_auc = roc_auc_score(y_valid_tree, custom_gbdt_valid_proba)
custom_gbdt_gini = 2 * custom_gbdt_auc - 1

print("Custom GBDT ROC AUC:", custom_gbdt_auc)
print("Custom GBDT Gini:", custom_gbdt_gini)

Custom GBDT ROC AUC: 0.6495312548328418
Custom GBDT Gini: 0.2990625096656836


Custom GBDT reached a validation Gini of `0.2991`.

It performs better than the custom single tree, but slightly worse than sklearn Random Forest.

In [16]:
# also a com table 

model_results = pd.DataFrame({
    "model": [
        "Custom Decision Tree",
        "Custom Random Forest",
        "Custom GBDT",
        "Sklearn Decision Tree",
        "Sklearn Random Forest"
    ],
    "validation_roc_auc": [
        custom_tree_auc,
        custom_rf_auc,
        custom_gbdt_auc,
        valid_auc_tree,
        valid_auc_sk_rf
    ],
    "validation_gini": [
        custom_tree_gini,
        custom_rf_gini,
        custom_gbdt_gini,
        valid_gini_tree,
        valid_gini_sk_rf
    ]
})

model_results.sort_values("validation_gini", ascending=False)

,model,validation_roc_auc,validation_gini
4,Sklearn Random Forest,0.658605,0.317209
2,Custom GBDT,0.649531,0.299063
3,Sklearn Decision Tree,0.647505,0.295009
1,Custom Random Forest,0.643755,0.287509
0,Custom Decision Tree,0.636033,0.272066


Sklearn Random Forest is still the best model so far.

Custom GBDT improved over the custom single tree, which shows that boosting helps, but it did not beat sklearn Random Forest.

In [19]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=21,
    n_jobs=-1
)

xgb_model.fit(X_train_prepared, y_train)

valid_proba_xgb = xgb_model.predict_proba(X_valid_prepared)[:, 1]

valid_auc_xgb = roc_auc_score(y_valid, valid_proba_xgb)
valid_gini_xgb = 2 * valid_auc_xgb - 1

print("XGBoost ROC AUC:", valid_auc_xgb)
print("XGBoost Gini:", valid_gini_xgb)

XGBoost ROC AUC: 0.6894952732197785
XGBoost Gini: 0.3789905464395571


XGBoost achieved a validation Gini score of `0.3790`.

It is the best model so far and performs better than the tree and random forest models.

In [21]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=21,
    n_jobs=-1
)

lgbm_model.fit(X_train_prepared, y_train)

valid_proba_lgbm = lgbm_model.predict_proba(X_valid_prepared)[:, 1]

valid_auc_lgbm = roc_auc_score(y_valid, valid_proba_lgbm)
valid_gini_lgbm = 2 * valid_auc_lgbm - 1

print("LightGBM ROC AUC:", valid_auc_lgbm)
print("LightGBM Gini:", valid_gini_lgbm)

[LightGBM] [Info] Number of positive: 2756, number of negative: 21328
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000892 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3962
[LightGBM] [Info] Number of data points in the train set: 24084, number of used features: 532
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.114433 -> initscore=-2.046240
[LightGBM] [Info] Start training from score -2.046240
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

/Users/abdo/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM achieved a validation Gini score of `0.3678`.

It performed well, but XGBoost is still the best model so far.

In [23]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=200,
    depth=4,
    learning_rate=0.05,
    loss_function="Logloss",
    random_seed=21,
    verbose=0
)

cat_model.fit(X_train_prepared, y_train)

valid_proba_cat = cat_model.predict_proba(X_valid_prepared)[:, 1]

valid_auc_cat = roc_auc_score(y_valid, valid_proba_cat)
valid_gini_cat = 2 * valid_auc_cat - 1

print("CatBoost ROC AUC:", valid_auc_cat)
print("CatBoost Gini:", valid_gini_cat)

CatBoost ROC AUC: 0.6710765965802351
CatBoost Gini: 0.3421531931604702


## CatBoost

CatBoost was trained as the third modern gradient boosting model for comparison.

CatBoost achieved a validation Gini score of `0.3422`.

It performed better than Random Forest, but worse than XGBoost and LightGBM in this setup.

In [25]:
# Table for comparison

all_models_results = pd.DataFrame({
    "model": [
        "Custom Decision Tree",
        "Custom Random Forest",
        "Custom GBDT",
        "Sklearn Decision Tree",
        "Sklearn Random Forest",
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],
    "validation_roc_auc": [
        custom_tree_auc,
        custom_rf_auc,
        custom_gbdt_auc,
        valid_auc_tree,
        valid_auc_sk_rf,
        valid_auc_xgb,
        valid_auc_lgbm,
        valid_auc_cat
    ],
    "validation_gini": [
        custom_tree_gini,
        custom_rf_gini,
        custom_gbdt_gini,
        valid_gini_tree,
        valid_gini_sk_rf,
        valid_gini_xgb,
        valid_gini_lgbm,
        valid_gini_cat
    ]
})

all_models_results.sort_values("validation_gini", ascending=False)

,model,validation_roc_auc,validation_gini
5,XGBoost,0.689495,0.378991
6,LightGBM,0.683921,0.367843
7,CatBoost,0.671077,0.342153
4,Sklearn Random Forest,0.658605,0.317209
2,Custom GBDT,0.649531,0.299063
3,Sklearn Decision Tree,0.647505,0.295009
1,Custom Random Forest,0.643755,0.287509
0,Custom Decision Tree,0.636033,0.272066


XGBoost achieved the best validation Gini score.

LightGBM was the second-best model, followed by CatBoost.

The modern boosting models performed better than single trees and random forests.

## Final Evaluation

XGBoost was selected as the best validation model.

The final step is to compare its Gini score on train, validation, and test data.

In [26]:
train_proba_xgb = xgb_model.predict_proba(X_train_prepared)[:, 1]
valid_proba_xgb = xgb_model.predict_proba(X_valid_prepared)[:, 1]
test_proba_xgb = xgb_model.predict_proba(X_test_prepared)[:, 1]

train_auc_xgb = roc_auc_score(y_train, train_proba_xgb)
valid_auc_xgb = roc_auc_score(y_valid, valid_proba_xgb)
test_auc_xgb = roc_auc_score(y_test, test_proba_xgb)

train_gini_xgb = 2 * train_auc_xgb - 1
valid_gini_xgb = 2 * valid_auc_xgb - 1
test_gini_xgb = 2 * test_auc_xgb - 1

final_xgb_results = pd.DataFrame({
    "dataset": ["Train", "Validation", "Test"],
    "roc_auc": [train_auc_xgb, valid_auc_xgb, test_auc_xgb],
    "gini": [train_gini_xgb, valid_gini_xgb, test_gini_xgb]
})

final_xgb_results

,dataset,roc_auc,gini
0,Train,0.780390,0.560780
1,Validation,0.689495,0.378991
2,Test,0.693810,0.387620


XGBoost achieved a validation Gini of `0.3790` and a test Gini of `0.3876`.

The test score is close to the validation score, so the model generalizes well to unseen data.

The train Gini is higher (`0.5608`), which suggests some overfitting, but the final test performance is still stable.

## Final Summary

XGBoost was the best model in this project.

It achieved the highest validation Gini score among the tested models and kept a stable performance on the test set.

Tree ensembles performed better than a single Decision Tree because they combine multiple trees and reduce the instability of individual trees.

## Final Conclusion

In this project, I compared custom and library-based tree models for bad buy prediction.

The experiments showed that:
- custom implementations helped explain the core logic of tree-based models
- ensemble methods improved performance over a single decision tree
- XGBoost achieved the best validation and test performance

This project helped me understand both the theory and the practical performance differences between custom ML implementations and optimized production-grade libraries.